# TensorRT FP32 baseline

Automatically benchmarks every `*_fp32.engine` under `original`, with its matching `.pt` checkpoint and VOC/KITTI validation split. Run all cells using the `notebook-env` kernel. Results and raw telemetry are saved under `PTQ/tensorrt_baseline_results`.

**Measurement definitions**

- Accuracy: full validation split, batch 1, 640 x 640 square input (`rect=False`). Recall is Ultralytics mean class recall at the validation-selected maximum mean F1 point. Latency uses `conf=0.25`, `iou=0.70`.
- Engine-only latency: CUDA-event duration of TensorRT `execute_async_v3`, with resident GPU input/output buffers and a representative validation image preprocessed before timing. No preprocessing, image upload, NMS, or disk access is counted. One inference is completed before the next begins.
- End-to-end latency: synchronized CPU wall time of Ultralytics `predict` on preloaded OpenCV images, including preprocessing, data transfer, TensorRT execution, NMS and result construction. Disk I/O, initialization and warmup are excluded. The first 100 sorted validation images are cycled for at least 15 seconds and 200 completed images.
- Each phase reports median/P95/mean latency. FPS from inverse mean latency and observed wall throughput are reported separately; engine-only CUDA-event FPS is theoretical service throughput, not application throughput.
- Power/temperature/clocks/utilization and GPU memory are sampled with NVML at approximately 100 ms. Idle power is measured for 5 seconds with the model resident but not executing, after 10 seconds of settling. GPU energy/image is integrated board power divided by completed images in the sustained test window. Incremental energy subtracts the corresponding idle power. These values cover the GPU board, not the full computer, and are window averages rather than energy of an isolated inference.
- NVML total GPU memory includes desktop and other processes. Load delta is approximate; process memory is reported when available (Windows WDDM may return unavailable). Peak memory is the highest sampled value and may miss shorter spikes. PyTorch allocated/reserved memory excludes allocations TensorRT makes outside PyTorch.
- Parameters and FLOPs describe the original PyTorch checkpoint at batch 1 and 640 x 640. THOP uses 2 FLOPs per MAC; these are reference-model operation counts, not TensorRT kernel instruction counts. File sizes use decimal MB.
- Models run sequentially. Background GPU work, thermal state, clock/power management and sampling resolution can affect the measurements. FP32 names identify the exported artifacts; TensorRT may permit TF32 internally.

Missing NVML fields remain `None`, not zero.


In [1]:
from pathlib import Path
import gc, json, os, platform, threading, time
from datetime import datetime, timezone
import cv2
import numpy as np
from IPython.display import display, HTML
import html
import torch
import tensorrt as trt
import pynvml as nv
import ultralytics
from ultralytics import YOLO, settings
import ultralytics.data.utils as data_utils
from ultralytics.utils.torch_utils import get_num_params, get_flops

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'original').is_dir())
OUTPUT = ROOT / 'PTQ/tensorrt_baseline_results'
OUTPUT.mkdir(parents=True, exist_ok=True)
DEVICE = 0
IMGSZ = 640
LATENCY_IMAGES = 100
WARMUP = 20
MIN_IMAGES = 200
RUN_SECONDS = 15.0
IDLE_SECONDS = 5.0
IDLE_SETTLE_SECONDS = 10.0
SAMPLE_SECONDS = 0.1
CONF, IOU = 0.25, 0.70
DATASETS = ROOT / 'PTQ/datasets'
settings.update({'datasets_dir': str(DATASETS)})
data_utils.DATASETS_DIR = DATASETS
assert torch.cuda.is_available(), 'CUDA-enabled PyTorch is required'
torch.cuda.set_device(DEVICE)
nv.nvmlInit()
gpu_uuid = torch.cuda.get_device_properties(DEVICE).uuid
GPU = nv.nvmlDeviceGetHandleByUUID(str(gpu_uuid))

def safe(fn, *args):
    try:
        value = fn(*args)
        return value if value < 2**63 else None
    except nv.NVMLError:
        return None

def memory_snapshot():
    info = nv.nvmlDeviceGetMemoryInfo(GPU)
    process = None
    try:
        matches = [p.usedGpuMemory for p in nv.nvmlDeviceGetComputeRunningProcesses(GPU)
                   if p.pid == os.getpid() and p.usedGpuMemory is not None and p.usedGpuMemory < 2**63]
        if matches:
            process = sum(matches) / 2**20
    except nv.NVMLError:
        pass
    return {'gpu_total_used_mib': info.used / 2**20, 'process_used_mib': process,
            'torch_allocated_mib': torch.cuda.memory_allocated(DEVICE) / 2**20,
            'torch_reserved_mib': torch.cuda.memory_reserved(DEVICE) / 2**20}

def sample_gpu():
    try:
        util = nv.nvmlDeviceGetUtilizationRates(GPU)
        gpu_util, memory_util = util.gpu, util.memory
    except nv.NVMLError:
        gpu_util = memory_util = None
    return {'t': time.perf_counter(), 'power_w': (p / 1000 if (p := safe(nv.nvmlDeviceGetPowerUsage, GPU)) is not None else None),
            'temperature_c': safe(nv.nvmlDeviceGetTemperature, GPU, nv.NVML_TEMPERATURE_GPU),
            'sm_clock_mhz': safe(nv.nvmlDeviceGetClockInfo, GPU, nv.NVML_CLOCK_SM),
            'memory_clock_mhz': safe(nv.nvmlDeviceGetClockInfo, GPU, nv.NVML_CLOCK_MEM),
            'gpu_util_percent': gpu_util, 'memory_util_percent': memory_util,
            **memory_snapshot()}

class Sampler:
    def __init__(self):
        self.rows, self.stop_event = [], threading.Event()
    def start(self):
        self.rows.append(sample_gpu())
        self.thread = threading.Thread(target=self.loop, daemon=True)
        self.thread.start()
        return self
    def loop(self):
        while not self.stop_event.wait(SAMPLE_SECONDS):
            self.rows.append(sample_gpu())
    def stop(self):
        self.stop_event.set()
        self.thread.join()
        self.rows.append(sample_gpu())
        return self.rows

def stats(values):
    x = np.asarray([v for v in values if v is not None], dtype=float)
    if not x.size:
        return {'mean': None, 'median': None, 'min': None, 'max': None}
    return {k: float(f(x)) for k, f in [('mean', np.mean), ('median', np.median), ('min', np.min), ('max', np.max)]}

def summarize_telemetry(rows, start, end, images, idle_power):
    power = [r for r in rows if r['power_w'] is not None]
    energy = None
    if len(power) >= 2:
        t = np.array([r['t'] for r in power])
        p = np.array([r['power_w'] for r in power])
        inner = t[(t > start) & (t < end)]
        knots = np.r_[start, inner, end]
        energy = float(np.trapezoid(np.interp(knots, t, p), knots))
    inside = [r for r in rows if start <= r['t'] <= end] or rows
    fields = ['power_w', 'temperature_c', 'sm_clock_mhz', 'memory_clock_mhz',
              'gpu_util_percent', 'memory_util_percent', 'gpu_total_used_mib', 'process_used_mib']
    result = {key: stats([r[key] for r in inside]) for key in fields}
    duration = end - start
    result.update({'sampling_interval_s': SAMPLE_SECONDS, 'samples': len(inside),
                   'duration_s': duration, 'images': images,
                   'mean_power_w_time_weighted': energy / duration if energy is not None else None,
                   'gpu_energy_j': energy,
                   'gpu_energy_j_per_image': energy / images if energy is not None and images else None,
                   'incremental_gpu_energy_j_per_image': (energy - idle_power * duration) / images
                      if energy is not None and idle_power is not None and images else None})
    return result

def idle_measure():
    torch.cuda.synchronize()
    time.sleep(IDLE_SETTLE_SECONDS)  # Allow clocks and NVML utilization counters to settle.
    sampler = Sampler().start()
    start = time.perf_counter()
    time.sleep(IDLE_SECONDS)
    end = time.perf_counter()
    rows = sampler.stop()
    summary = summarize_telemetry(rows, start, end, 0, None)
    return summary['mean_power_w_time_weighted'], summary, rows

def latency_summary(ms, wall_seconds):
    x = np.asarray(ms)
    return {'images': len(ms), 'mean_ms': float(x.mean()), 'median_ms': float(np.median(x)),
            'p95_ms': float(np.percentile(x, 95)), 'fps_inverse_mean_latency': float(1000 / x.mean()),
            'wall_duration_s': wall_seconds, 'fps_observed_wall': len(ms) / wall_seconds}

def load_engine(path):
    logger = trt.Logger(trt.Logger.ERROR)
    runtime = trt.Runtime(logger)
    with path.open('rb') as f:
        n = int.from_bytes(f.read(4), 'little')
        metadata = {}
        offset = 0
        if 0 < n < path.stat().st_size - 4:
            try:
                metadata = json.loads(f.read(n))
                offset = n + 4
            except (ValueError, UnicodeDecodeError):
                pass
        f.seek(offset)
        engine = runtime.deserialize_cuda_engine(f.read())
    if engine is None:
        raise RuntimeError(f'Failed to deserialize {path}')
    context = engine.create_execution_context()
    buffers = {}
    dtypes = {trt.float32: torch.float32, trt.float16: torch.float16, trt.int32: torch.int32,
              trt.int64: torch.int64, trt.int8: torch.int8, trt.bool: torch.bool}
    for i in range(engine.num_io_tensors):
        name = engine.get_tensor_name(i)
        shape = tuple(engine.get_tensor_shape(name))
        if any(d < 0 for d in shape):
            raise RuntimeError('This baseline requires static batch-1 engines')
        buffers[name] = torch.empty(shape, device=f'cuda:{DEVICE}', dtype=dtypes[engine.get_tensor_dtype(name)])
        context.set_tensor_address(name, buffers[name].data_ptr())
    return runtime, engine, context, buffers, metadata

def clean_cuda():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

def engine_only(path, image):
    clean_cuda()
    before = memory_snapshot()
    runtime, engine, context, buffers, metadata = load_engine(path)
    from ultralytics.data.augment import LetterBox
    prepared = LetterBox((IMGSZ, IMGSZ), auto=False)(image=image)
    array = np.ascontiguousarray(prepared[:, :, ::-1].transpose(2, 0, 1)[None])
    input_names = [name for name in buffers if engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT]
    if len(input_names) != 1:
        raise RuntimeError('Expected a single image input')
    buffers[input_names[0]].copy_(torch.from_numpy(array).to(DEVICE).to(buffers[input_names[0]].dtype) / 255)
    stream = torch.cuda.current_stream(DEVICE)
    for _ in range(WARMUP):
        if not context.execute_async_v3(stream.cuda_stream):
            raise RuntimeError('TensorRT enqueue failed')
    torch.cuda.synchronize()
    loaded = memory_snapshot()
    idle_power, idle, idle_rows = idle_measure()
    begin, finish = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    timings = []
    sampler = Sampler().start()
    start = time.perf_counter()
    try:
        while len(timings) < MIN_IMAGES or time.perf_counter() - start < RUN_SECONDS:
            begin.record(stream)
            ok = context.execute_async_v3(stream.cuda_stream)
            finish.record(stream)
            finish.synchronize()
            if not ok:
                raise RuntimeError('TensorRT enqueue failed')
            timings.append(begin.elapsed_time(finish))
        end = time.perf_counter()
    finally:
        rows = sampler.stop()
    result = {'latency': latency_summary(timings, end-start), 'memory_before_load': before,
              'memory_model_loaded': loaded, 'memory_after_inference': memory_snapshot(),
              'idle_power_w': idle_power, 'idle_telemetry': idle,
              'telemetry': summarize_telemetry(rows, start, end, len(timings), idle_power),
              'engine_metadata': metadata,
              'io_types': {name: str(engine.get_tensor_dtype(name)) for name in buffers}}
    del context, buffers, engine, runtime
    clean_cuda()
    return result, {'idle': idle_rows, 'running': rows, 'latency_ms': timings}

def end_to_end(path, images):
    clean_cuda()
    before = memory_snapshot()
    model = YOLO(str(path), task='detect')
    kwargs = dict(imgsz=IMGSZ, device=DEVICE, conf=CONF, iou=IOU, rect=False,
                  save=False, verbose=False)
    for i in range(WARMUP):
        model.predict(images[i % len(images)], **kwargs)
    torch.cuda.synchronize()
    loaded = memory_snapshot()
    idle_power, idle, idle_rows = idle_measure()
    timings = []
    sampler = Sampler().start()
    start = time.perf_counter()
    try:
        while len(timings) < MIN_IMAGES or time.perf_counter() - start < RUN_SECONDS:
            torch.cuda.synchronize()
            begin = time.perf_counter()
            prediction = model.predict(images[len(timings) % len(images)], **kwargs)
            torch.cuda.synchronize()
            timings.append((time.perf_counter()-begin)*1000)
            del prediction
        end = time.perf_counter()
    finally:
        rows = sampler.stop()
    result = {'latency': latency_summary(timings, end-start), 'memory_before_load': before,
              'memory_model_loaded': loaded, 'memory_after_inference': memory_snapshot(),
              'idle_power_w': idle_power, 'idle_telemetry': idle,
              'telemetry': summarize_telemetry(rows, start, end, len(timings), idle_power)}
    del model
    clean_cuda()
    return result, {'idle': idle_rows, 'running': rows, 'latency_ms': timings}

def discover():
    engines = sorted(p for p in (ROOT/'original').rglob('*.engine') if p.name.lower().endswith('_fp32.engine'))
    configs = []
    for engine in engines:
        weights = engine.with_name(engine.name[:-len('_fp32.engine')] + '.pt')
        folder = engine.parent.name.lower()
        if 'voc' in folder:
            data, images = 'VOC.yaml', DATASETS/'VOC/images/test2007'
        elif 'kitti' in folder:
            data, images = 'kitti.yaml', DATASETS/'kitti/images/val'
        else:
            raise ValueError(f'Cannot infer dataset for {engine}')
        if not weights.is_file() or not images.is_dir():
            raise FileNotFoundError(f'Missing weights or dataset: {weights}, {images}')
        configs.append({'engine': engine, 'weights': weights, 'data': data, 'images': images})
    if not configs:
        raise FileNotFoundError('No *_fp32.engine models in original')
    return configs

def benchmark_model(config):
    engine, weights = config['engine'], config['weights']
    print(f'\nBENCHMARK: {engine.name}', flush=True)
    paths = sorted(p for p in config['images'].iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png'})[:LATENCY_IMAGES]
    images = [cv2.imread(str(p)) for p in paths]
    if not images or any(im is None for im in images):
        raise RuntimeError('Cannot preload validation images')
    reference = YOLO(str(weights), task='detect')
    params = int(get_num_params(reference.model))
    gflops = float(get_flops(reference.model, IMGSZ))
    if not gflops:
        raise RuntimeError('THOP FLOPs profiling failed')
    del reference
    clean_cuda()
    print('Running full validation split...', flush=True)
    detector = YOLO(str(engine), task='detect')
    metrics = detector.val(data=config['data'], split='val', imgsz=IMGSZ, batch=1, device=DEVICE,
                           rect=False, workers=0, plots=False, save_json=False, verbose=False,
                           project=str(OUTPUT/'validation'), name=engine.stem, exist_ok=True)
    accuracy = {'map50_95': float(metrics.box.map), 'map50': float(metrics.box.map50),
                'recall': float(metrics.box.mr),
                'per_class_recall': np.asarray(metrics.box.r).tolist(),
                'recall_definition': 'Ultralytics mean class recall at its validation-selected max-mean-F1 operating point'}
    del detector, metrics
    clean_cuda()
    print('Measuring engine-only latency and GPU telemetry...', flush=True)
    engine_result, engine_samples = engine_only(engine, images[0])
    print('Measuring end-to-end latency and GPU telemetry...', flush=True)
    e2e_result, e2e_samples = end_to_end(engine, images)
    result = {'model': str(engine.relative_to(ROOT)), 'weights': str(weights.relative_to(ROOT)),
              'dataset': config['data'], 'accuracy': accuracy, 'parameters': params,
              'gflops': gflops, 'gmacs': gflops/2, 'weights_size_mb': weights.stat().st_size/1e6,
              'engine_size_mb': engine.stat().st_size/1e6, 'engine_only': engine_result,
              'end_to_end': e2e_result, 'latency_image_count': len(images)}
    (OUTPUT/f'{engine.stem}_samples.json').write_text(json.dumps({'engine_only':engine_samples,'end_to_end':e2e_samples}, indent=2), encoding='utf-8')
    (OUTPUT/f'{engine.stem}.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(json.dumps({'model': engine.name, **accuracy,
                      'engine_median_ms':engine_result['latency']['median_ms'],
                      'e2e_median_ms':e2e_result['latency']['median_ms']}, indent=2), flush=True)
    return result

CONFIGS = discover()
ENVIRONMENT = {'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'gpu': torch.cuda.get_device_name(DEVICE),
               'gpu_uuid': str(gpu_uuid), 'python': platform.python_version(), 'torch':torch.__version__,
               'tensorrt':trt.__version__, 'ultralytics':ultralytics.__version__,
               'driver':nv.nvmlSystemGetDriverVersion(), 'imgsz':IMGSZ, 'batch':1,
               'run_seconds_min':RUN_SECONDS, 'idle_seconds':IDLE_SECONDS, 'warmup':WARMUP,
               'idle_settle_seconds':IDLE_SETTLE_SECONDS,
               'sampling_interval_s':SAMPLE_SECONDS, 'pid':os.getpid()}
print(json.dumps(ENVIRONMENT, indent=2))
print('FP32 engines:', [str(c['engine'].relative_to(ROOT)) for c in CONFIGS])


{
  "timestamp_utc": "2026-09-30T13:26:31.143872+00:00",
  "gpu": "NVIDIA RTX A2000",
  "gpu_uuid": "a0aa0f21-a2f1-4018-b27b-236640f36d02",
  "python": "3.14.7",
  "torch": "2.14.0+cu130",
  "tensorrt": "11.3.0.99",
  "ultralytics": "8.4.163",
  "driver": "581.29",
  "imgsz": 640,
  "batch": 1,
  "run_seconds_min": 15.0,
  "idle_seconds": 5.0,
  "warmup": 20,
  "idle_settle_seconds": 10.0,
  "sampling_interval_s": 0.1,
  "pid": 19220
}
FP32 engines: ['original\\yolov8n_kitti\\8n_float32_fp32.engine', 'original\\yolov8n_voc\\voc_8n_best_fp32.engine']


In [2]:
RESULTS = []
for config in CONFIGS:
    result = benchmark_model(config)
    RESULTS.append(result)
    (OUTPUT/'results.json').write_text(json.dumps({'environment': ENVIRONMENT, 'models': RESULTS}, indent=2), encoding='utf-8')
print('Saved:', OUTPUT/'results.json')


Remeasuring latency and telemetry after idle settling: 8n_float32_fp32.engine


Loading C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_kitti\8n_float32_fp32.engine for TensorRT inference...


Completed: 8n_float32_fp32.engine


Remeasuring latency and telemetry after idle settling: voc_8n_best_fp32.engine


Loading C:\Users\dxu1\Desktop\Dissertation\original\yolov8n_voc\voc_8n_best_fp32.engine for TensorRT inference...


Completed: voc_8n_best_fp32.engine


Saved: C:\Users\dxu1\Desktop\Dissertation\PTQ\tensorrt_baseline_results\results.json


In [3]:
from pathlib import Path
import json
import html
from IPython.display import display, HTML, Markdown

project_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'original').is_dir())
output_dir = project_root/'PTQ/tensorrt_baseline_results'
saved = json.loads((output_dir/'results.json').read_text(encoding='utf-8'))
records = saved['models']
environment = saved['environment']
accuracy_speed_rows, resource_energy_rows = [], []

for r in records:
    model = 'YOLOv8n' if '8n' in Path(r['model']).name else 'YOLOv8l' if '8l' in Path(r['model']).name else Path(r['model']).stem
    dataset = Path(r['dataset']).stem.upper()
    engine_latency = r['engine_only']['latency']
    e2e = r['end_to_end']
    latency = e2e['latency']
    telemetry = e2e['telemetry']
    accuracy_speed_rows.append({
        'Model': model, 'Dataset': dataset, 'Precision': 'FP32',
        'mAP50-95 (%)': r['accuracy']['map50_95']*100,
        'mAP50 (%)': r['accuracy']['map50']*100,
        'Engine median / P95 (ms)': f"{engine_latency['median_ms']:.3f} / {engine_latency['p95_ms']:.3f}",
        'E2E median / P95 (ms)': f"{latency['median_ms']:.3f} / {latency['p95_ms']:.3f}",
        'E2E FPS': latency['fps_observed_wall'],
    })
    resource_energy_rows.append({
        'Model': model, 'Dataset': dataset,
        'Params (M)': r['parameters']/1e6, 'GFLOPs': r['gflops'],
        'Weights (MB)': r['weights_size_mb'], 'Engine (MB)': r['engine_size_mb'],
        'Approx. GPU load delta (MiB)': e2e['memory_model_loaded']['gpu_total_used_mib']-e2e['memory_before_load']['gpu_total_used_mib'],
        'E2E GPU power (W)': telemetry['mean_power_w_time_weighted'],
        'E2E GPU energy (J/image)': telemetry['gpu_energy_j_per_image'],
    })

def format_value(key, value):
    if value is None:
        return 'N/A'
    if isinstance(value, float):
        decimals = 4 if 'J/image' in key else 3 if key == 'Params (M)' else 1 if key == 'E2E FPS' else 2
        return f'{value:.{decimals}f}'
    return str(value)

def render_table(rows):
    keys = list(rows[0])
    header = ''.join('<th>'+html.escape(k)+'</th>' for k in keys)
    body = ''.join('<tr>'+''.join('<td>'+html.escape(format_value(k,row[k]))+'</td>' for k in keys)+'</tr>' for row in rows)
    display(HTML('<div style="overflow-x:auto"><table><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table></div>'))

def markdown_table(rows):
    keys = list(rows[0])
    return '\n'.join(['| '+' | '.join(keys)+' |', '| '+' | '.join(['---']*len(keys))+' |',
                      *['| '+' | '.join(format_value(k,row[k]) for k in keys)+' |' for row in rows]])

notes1 = f"Batch = 1; input = {environment['imgsz']} x {environment['imgsz']}; GPU = {environment['gpu']}. mAP is reported as a percentage. Engine latency uses CUDA events and resident GPU buffers. E2E includes preprocessing, data transfer, inference, NMS and result construction; disk I/O is excluded. FPS is completed images divided by measured wall time, not the reciprocal of median latency. P95 describes per-image latency within a run."
notes2 = 'Parameters and GFLOPs describe the original PyTorch checkpoint; 1 MAC is counted as 2 FLOPs. File sizes use decimal MB. GPU load delta uses total-device NVML memory before loading versus after warmup and is approximate, not process-specific model memory. Power and energy use the E2E sustained test; energy includes idle GPU power and covers the GPU board, not the full computer. Results are single-run measurements; repeat under comparable thermal conditions before final publication.'

display(Markdown('**Table 1. FP32 baseline accuracy and inference performance**'))
render_table(accuracy_speed_rows)
display(Markdown(notes1))
display(Markdown('**Table 2. FP32 baseline deployment resources and GPU energy**'))
render_table(resource_energy_rows)
display(Markdown(notes2))

tables = {'accuracy_speed': accuracy_speed_rows, 'resources_energy': resource_energy_rows,
          'notes': {'table1': notes1, 'table2': notes2}, 'environment': environment}
(output_dir/'baseline_tables.json').write_text(json.dumps(tables, indent=2), encoding='utf-8')
(output_dir/'baseline_tables.md').write_text(
    '# Table 1. FP32 baseline accuracy and inference performance\n\n'+markdown_table(accuracy_speed_rows)+'\n\n'+notes1+
    '\n\n# Table 2. FP32 baseline deployment resources and GPU energy\n\n'+markdown_table(resource_energy_rows)+'\n\n'+notes2+'\n', encoding='utf-8')
print('Saved paper tables:', output_dir/'baseline_tables.md')


**Table 1. FP32 baseline accuracy and inference performance**

Model,Dataset,Precision,mAP50-95 (%),mAP50 (%),Engine median / P95 (ms),E2E median / P95 (ms),E2E FPS
YOLOv8n,KITTI,FP32,67.55,90.49,2.623 / 3.151,4.264 / 5.625,216.8
YOLOv8n,VOC,FP32,62.13,82.51,2.681 / 3.232,4.377 / 5.493,216.8


Batch = 1; input = 640 x 640; GPU = NVIDIA RTX A2000. mAP is reported as a percentage. Engine latency uses CUDA events and resident GPU buffers. E2E includes preprocessing, data transfer, inference, NMS and result construction; disk I/O is excluded. FPS is completed images divided by measured wall time, not the reciprocal of median latency. P95 describes per-image latency within a run.

**Table 2. FP32 baseline deployment resources and GPU energy**

Model,Dataset,Params (M),GFLOPs,Weights (MB),Engine (MB),Approx. GPU load delta (MiB),E2E GPU power (W),E2E GPU energy (J/image)
YOLOv8n,KITTI,3.012,8.20,6.24,79.82,147.19,63.91,0.2947
YOLOv8n,VOC,3.015,8.21,6.26,93.51,138.00,63.84,0.2945


Parameters and GFLOPs describe the original PyTorch checkpoint; 1 MAC is counted as 2 FLOPs. File sizes use decimal MB. GPU load delta uses total-device NVML memory before loading versus after warmup and is approximate, not process-specific model memory. Power and energy use the E2E sustained test; energy includes idle GPU power and covers the GPU board, not the full computer. Results are single-run measurements; repeat under comparable thermal conditions before final publication.

Saved paper tables: C:\Users\dxu1\Desktop\Dissertation\PTQ\tensorrt_baseline_results\baseline_tables.md
